# 1. Contexte du projet


# Analyse des ventes de la librairie Lapage

## Contexte

Ce notebook a été réalisé dans le cadre du projet OpenClassrooms Data Analyst.

L'objectif est d'analyser les ventes de la librairie Lapage afin de mieux comprendre l'activité commerciale, le comportement des clients et les performances des différentes catégories de produits.

Les analyses permettront notamment d'identifier les tendances de ventes, les profils de clientèle et les principaux indicateurs commerciaux.

## Jeux de données

L'analyse repose sur trois fichiers :

- **transactions.csv** : historique des ventes réalisées entre mars 2021 et février 2023.
- **products.csv** : informations sur les produits (prix et catégorie).
- **customers.csv** : informations sur les clients (identifiant, sexe et année de naissance).

Ces trois jeux de données seront nettoyés puis fusionnés afin d'obtenir une base unique utilisée pour l'ensemble des analyses.

# 2. Importation des bibliothèques


pip install pandas numpy matplotlib seaborn scipy plotly statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 3. Chargement des données

In [ ]:
customers = pd.read_csv("customers.csv", sep=";")
products = pd.read_csv("products.csv", sep=";")
transactions = pd.read_csv("Transactions.csv", sep=";", low_memory=False)

# 4. Contrôle qualité des données


In [ ]:
customers.head()

In [ ]:
customers.info()

In [ ]:
products.info()

In [ ]:
transactions.info()

Cette étape permet d'identifier les données manquantes susceptibles de perturber les analyses.

## 4.1 Valeurs manquantes

In [ ]:
customers.isnull().sum()

In [ ]:
products.isnull().sum()

In [ ]:
transactions.isnull().sum()

### Interprétation

Les tables customers et products ne présentent aucune valeur manquante.

La table transactions contient 361 041 lignes entièrement vides. Elles sont supprimées avant les analyses.

In [ ]:
transactions = transactions.dropna(how='all')

In [ ]:
transactions.info()

In [ ]:
transactions.info()

## 4.2 Doublons

In [ ]:
customers.duplicated().sum()  

In [ ]:
products.duplicated().sum()

Nous vérifions si les jeux de données contiennent des doublons afin de garantir la qualité des données avant de réaliser les analyses.

In [ ]:
transactions.duplicated().sum()

### Interprétation

Aucun doublon n'a été détecté dans les trois tables.

Les données peuvent donc être utilisées sans suppression supplémentaire de lignes.

## 4.3 Conversion de la colonne date

In [ ]:
transactions["date"] = pd.to_datetime(transactions["date"])

transactions.info()

### Interprétation

La colonne **date** a été convertie avec succès au format **datetime64[ns]**. Cette transformation permettra de réaliser des analyses temporelles (évolution des ventes par année, mois, trimestre ou jour) au cours du projet.

## 4.4 Analyse descriptive des variables numériques

In [ ]:
products.describe()

### Interprétation

Les prix des produits varient de **0,62 € à 300 €**.

Aucune valeur négative ou incohérente n'a été détectée. Les statistiques descriptives montrent une forte dispersion des prix, ce qui est cohérent avec un catalogue composé de différentes catégories de produits.

In [ ]:
customers["birth"].describe()

## Interprétation

Les années de naissance sont comprises entre **1929** et **2004**.

Ces valeurs sont cohérentes et ne présentent pas d'anomalies apparentes. Elles pourront être utilisées pour calculer l'âge des clients.

Les données couvrent plusieurs générations de clients, ce qui permettra d'étudier le comportement d'achat selon l'âge.

## 4.4.1 Distribution des années de naissance

In [ ]:
plt.figure(figsize=(8,5))

plt.boxplot(customers["birth"])

plt.title("Distribution des années de naissance")
plt.ylabel("Année")

plt.show()

## Interprétation

La distribution des années de naissance est relativement homogène. La majorité des clients est née entre les années **1966 et 1992**, avec une médiane proche de **1979**. Aucun point aberrant n'est observé, ce qui confirme la cohérence des données utilisées pour le calcul de l'âge.

# 5. Fusion des données

Les analyses de ventes nécessitent de regrouper les informations provenant **des trois tables (transactions, produits et clients)** afin d'obtenir un jeu de données unique.

In [ ]:
transactions_products = transactions.merge(
    products,
    on="id_prod",
    how="left"
)

transactions_products.head()

In [ ]:
transactions_final = transactions_products.merge(
    customers,
    on="client_id",
    how="left"
)

transactions_final.head()

In [ ]:
transactions_final.info()

In [ ]:
transactions_final.shape

## Interprétation

Les trois tables ont été fusionnées avec succès afin de constituer un jeu de données unique.

Le jeu de données final contient **687 534 transactions** et **8 variables**, sans perte d'information lors des jointures.

Cette table sera utilisée pour réaliser l'ensemble des analyses de ventes et des analyses statistiques demandées dans le projet.

Cette partie a pour objectif **d'étudier les principaux indicateurs de performance commerciale de la librairie** afin **de mieux comprendre l'activité de vente**.

# 6.1 Indicateurs commerciaux


Une ligne du jeu de données correspond à un produit acheté au cours d'une transaction.

Il est donc nécessaire de distinguer :

- le **nombre d'articles vendus**, correspondant au nombre de lignes ;
- le **nombre de commandes**, correspondant au nombre de sessions uniques ;
- le **panier moyen**, correspondant au montant moyen dépensé par session.

Le prix moyen d'un produit ne doit pas être confondu avec le panier moyen d'une commande.

In [ ]:
# Calcul du chiffre d'affaires de chaque commande

commandes = (
    transactions_final
    .groupby("session_id", as_index=False)
    .agg(
        montant_commande=("price", "sum"),
        nombre_articles=("id_prod", "count"),
        client_id=("client_id", "first"),
        date_commande=("date", "min")
    )
)

commandes.head()

In [ ]:
# Calcul des principaux indicateurs commerciaux

ca_total = transactions_final["price"].sum()

nombre_articles = len(transactions_final)

nombre_commandes = transactions_final["session_id"].nunique()

nombre_clients_acheteurs = transactions_final["client_id"].nunique()

panier_moyen = commandes["montant_commande"].mean()

articles_moyens_par_commande = commandes["nombre_articles"].mean()

print(f"Chiffre d'affaires total : {ca_total:,.2f} €")
print(f"Nombre d'articles vendus : {nombre_articles:,}")
print(f"Nombre de commandes : {nombre_commandes:,}")
print(f"Nombre de clients acheteurs : {nombre_clients_acheteurs:,}")
print(f"Panier moyen par commande : {panier_moyen:,.2f} €")
print(f"Nombre moyen d'articles par commande : {articles_moyens_par_commande:.2f}")

### Interprétation

Les principaux indicateurs commerciaux montrent que la librairie a réalisé un chiffre d'affaires total de **12 027 663,10 €** sur la période étudiée.

Au total, **687 534 articles** ont été vendus au cours de **345 505 commandes**, effectuées par **8 600 clients**.

Le **panier moyen** s'élève à **34,81 € par commande**, avec **1,99 article acheté en moyenne par commande**.

Ces résultats montrent que la majorité des commandes comportent environ deux articles, ce qui constitue une base pertinente pour les analyses détaillées des ventes et du comportement des clients.

In [ ]:
# Vérification : le CA des commandes doit être égal au CA total

ca_commandes = commandes["montant_commande"].sum()

print(f"CA calculé depuis les transactions : {ca_total:,.2f} €")
print(f"CA calculé depuis les commandes : {ca_commandes:,.2f} €")
print(f"Écart : {ca_total - ca_commandes:.2f} €")

### Interprétation

Les principaux indicateurs commerciaux montrent que la librairie a généré un chiffre d'affaires total de **12 027 663 €** sur la période étudiée.

Au total, **687 534 articles** ont été vendus au travers de **345 505 commandes**, soit près de **2 articles par commande** (1,99 en moyenne).

Le **panier moyen** s'élève à **34,81 € par commande**.

Cette valeur est nettement supérieure au prix moyen d'un article (17,49 €), ce qui confirme qu'une commande contient généralement plusieurs produits.

Ces indicateurs fournissent une vision plus fidèle de l'activité commerciale que le simple prix moyen des produits.

## 6.2 Évolution du chiffre d'affaires par mois

Nous analysons l'évolution du chiffre d'affaires au fil des mois afin d'identifier les tendances de ventes et les éventuelles périodes de forte ou faible activité.

In [ ]:
# Calcul du chiffre d'affaires mensuel

ca_mensuel = (
    transactions_final
    .groupby(transactions_final["date"].dt.to_period("M"))["price"]
    .sum()
)

ca_mensuel.head()

In [ ]:
import plotly.express as px

fig = px.line(
    x=ca_mensuel.index.astype(str),
    y=ca_mensuel.values,
    markers=True,
    labels={
        "x":"Mois",
        "y":"Chiffre d'affaires (€)"
    },
    title="Évolution du chiffre d'affaires mensuel"
)

fig.update_traces(
    hovertemplate="<b>%{x}</b><br>CA : %{y:,.0f} €<extra></extra>"
)

fig.update_layout(
    xaxis_title="Mois",
    yaxis_title="Chiffre d'affaires (€)",
    hovermode="x unified"
)

fig.show()

### Interprétation

Le chiffre d'affaires mensuel évolue entre environ **475 000 €** et **535 000 €**.

On observe une progression régulière jusqu'au début de l'année **2022**, avec un pic de chiffre d'affaires enregistré en **février 2022**.

À partir du printemps 2022, le chiffre d'affaires reste relativement stable autour de **500 000 € par mois**, malgré quelques variations normales de l'activité commerciale.

**Note :** février 2023 n'est **pas** un mois incomplet (données jusqu'au 28/02/2023). Les variations observées en fin de période doivent donc être interprétées comme de réelles fluctuations d'activité, et non comme un artefact de collecte.


## 6.2 bis Série temporelle du CA quotidien + moyenne mobile

Le CA quotidien est bruité : on le lisse par une **moyenne mobile à 30 jours** pour faire apparaître la tendance.


In [ ]:
ca_jour = transactions_final.groupby(transactions_final['date'].dt.date)['price'].sum()
ca_jour.index = pd.to_datetime(ca_jour.index)
mm30 = ca_jour.rolling(30, center=True).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ca_jour.index, ca_jour.values, lw=0.6, alpha=0.4, label='CA quotidien')
ax.plot(mm30.index, mm30.values, lw=2.2, color='crimson', label='Moyenne mobile 30 j')
ax.set_title("CA quotidien et moyenne mobile 30 jours")
ax.set_ylabel('CA (€)')
ax.legend()
plt.tight_layout()
plt.show()


## 6.2 ter Décomposition de la série (tendance / saisonnalité / résidus)

Modèle additif : $$CA_t = Tendance_t + Saisonnalité_t + Résidu_t$$


In [ ]:
tendance = ca_jour.rolling(30, center=True).mean()
detrend = ca_jour - tendance
saison = detrend.groupby(detrend.index.dayofweek).transform('mean')
residu = ca_jour - tendance - saison

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
axes[0].plot(ca_jour, lw=0.7); axes[0].set_title('Série observée (CA quotidien)')
axes[1].plot(tendance, lw=2, color='crimson'); axes[1].set_title('Tendance (MM 30 j)')
axes[2].plot(saison, lw=0.7, color='seagreen'); axes[2].set_title('Saisonnalité hebdomadaire')
axes[3].plot(residu, lw=0.5, color='grey'); axes[3].set_title('Résidus')
plt.tight_layout()
plt.show()

prof = detrend.groupby(detrend.index.dayofweek).mean()
prof.index = ['Lun','Mar','Mer','Jeu','Ven','Sam','Dim']
print('Effet moyen par jour (€) :')
print(prof.round(0).to_string())
print(f"Amplitude saisonnière : {prof.max()-prof.min():,.0f} € (CA moyen/jour ≈ {ca_jour.mean():,.0f} €)")


> **Lecture :** l'amplitude saisonnière hebdomadaire est très faible (< 1 % du CA journalier). Pas d'effet jour-de-semaine marqué ; l'essentiel des variations vient de la tendance et d'événements ponctuels (ex. automne 2021).


## 6.3 Évolution du nombre de ventes par mois

Nous analysons le nombre de ventes réalisées chaque mois afin d'observer le rythme de l'activité commerciale et d'identifier les éventuelles variations de volume au cours du temps.

In [ ]:
# Calcul du nombre de ventes par mois

ventes_mensuelles = (
    transactions_final
    .groupby(transactions_final["date"].dt.to_period("M"))
    .size()
)

ventes_mensuelles.head()

In [ ]:
import plotly.express as px

fig = px.line(
    x=ventes_mensuelles.index.astype(str),
    y=ventes_mensuelles.values,
    markers=True,
    labels={
        "x": "Mois",
        "y": "Nombre de ventes"
    },
    title="Évolution du nombre de ventes par mois"
)

fig.update_traces(
    hovertemplate="<b>%{x}</b><br>Ventes : %{y:,.0f}<extra></extra>"
)

fig.update_layout(
    xaxis_title="Mois",
    yaxis_title="Nombre de ventes",
    hovermode="x unified"
)

fig.show()

### Interprétation

Le nombre de ventes varie au cours de la période étudiée, avec un volume généralement compris entre **25 000 et 33 000 ventes par mois**.

Après une baisse progressive au cours du premier semestre 2021, le volume de ventes atteint son niveau le plus bas durant l'été 2021, avant de connaître une forte reprise à partir de septembre 2021.

Durant l'année 2022, l'activité se stabilise globalement autour de **28 000 à 30 000 ventes mensuelles**, malgré quelques fluctuations ponctuelles.

**Note :** février 2023 est un mois **complet** (jusqu'au 28/02/2023). Toute baisse observée sur ce mois ne s'explique pas par un mois tronqué dans le jeu de données.


## 6.4 Chiffre d'affaires par catégorie

Nous analysons le chiffre d'affaires généré par chaque catégorie de produits afin d'identifier celles qui contribuent le plus aux revenus de la librairie.

In [ ]:
# Calcul du chiffre d'affaires par catégorie

ca_categorie = (
    transactions_final
    .groupby("categ")["price"]
    .sum()
)

In [ ]:
import plotly.express as px

fig = px.bar(
    x=ca_categorie.index.map(lambda x: f"Catégorie {x}"),
    y=ca_categorie.values,
    text=ca_categorie.values,
    labels={
        "x":"Catégorie",
        "y":"Chiffre d'affaires (€)"
    },
    title="Chiffre d'affaires par catégorie"
)

fig.update_traces(
    texttemplate='%{text:,.0f} €',
    textposition='outside'
)

fig.update_layout(
    xaxis_title="Catégorie",
    yaxis_title="Chiffre d'affaires (€)",
   yaxis_range=[0, ca_categorie.max() * 1.15]
)

fig.show()

### Interprétation

Le chiffre d'affaires est réparti de manière inégale entre les trois catégories de produits.

- **La catégorie 1** génère le **chiffre d'affaires le plus élevé**, avec environ **4 827 657 €**. Elle constitue donc la principale source de revenus de la librairie.

- **La catégorie 0** arrive en deuxième position avec un chiffre d'affaires d'environ **4 419 731 €**, ce qui montre qu'elle contribue également de manière importante aux ventes.

- **La catégorie 2** enregistre le **chiffre d'affaires le plus faible**, avec environ **2 780 275 €**, soit nettement moins que les deux autres catégories.

Ces résultats montrent que les **catégories 0 et 1 représentent l'essentiel du chiffre d'affaires** de la librairie, tandis que la catégorie 2 contribue plus modestement aux revenus.

Dans la section suivante, nous analyserons le **nombre de ventes par catégorie** afin de déterminer si ces écarts de chiffre d'affaires s'expliquent par un volume de ventes plus important, par un prix moyen plus élevé ou par une combinaison de ces deux facteurs.

## 6.5 Nombre de ventes par catégorie

Nous analysons le volume de ventes réalisé pour chaque catégorie de produits afin d'identifier les catégories les plus vendues.

In [ ]:
# Nombre de ventes par catégorie

ventes_categorie = (
    transactions_final
    .groupby("categ")
    .size()
)

ventes_categorie

In [ ]:
import plotly.express as px

fig = px.bar(
    x=ventes_categorie.index.map(lambda x: f"Catégorie {x}"),
    y=ventes_categorie.values,
    text=ventes_categorie.values,
    labels={
        "x": "Catégorie",
        "y": "Nombre de ventes"
    },
    title="Nombre de ventes par catégorie"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Nombre de ventes : %{y:,.0f}<extra></extra>"
)

fig.update_layout(
    xaxis_title="Catégorie",
    yaxis_title="Nombre de ventes",
    yaxis_range=[0, ventes_categorie.max() * 1.15]
)

fig.show()

### Interprétation

Le volume de ventes est très différent selon les catégories de produits.

La catégorie **0** est la plus vendue avec **415 459 ventes**, soit près de 60 % de l'ensemble des ventes.

La catégorie **1** arrive en deuxième position avec **235 592 ventes**, tandis que la catégorie **2** représente une part beaucoup plus faible avec seulement **36 483 ventes**.

Ces résultats montrent que l'activité commerciale repose principalement sur les catégories **0** et **1**, la catégorie **0** étant la plus populaire en termes de volume de ventes.

## 6.6 Chiffre d'affaires par sexe

Nous analysons le chiffre d'affaires généré par les clients selon leur sexe afin d'identifier les éventuelles différences de contribution entre les hommes et les femmes.

In [ ]:
# Calcul du chiffre d'affaires par sexe

ca_sexe = (
    transactions_final
    .groupby("sex")["price"]
    .sum()
)

ca_sexe

In [ ]:
import plotly.express as px

# Renommer les modalités pour améliorer la lisibilité
labels_sexe = {
    "f": "Femmes",
    "m": "Hommes"
}

fig = px.bar(
    x=ca_sexe.index.map(labels_sexe),
    y=ca_sexe.values,
    text=ca_sexe.values,
    labels={
        "x": "Sexe",
        "y": "Chiffre d'affaires (€)"
    },
    title="Chiffre d'affaires par sexe"
)

fig.update_traces(
    texttemplate="%{text:,.0f} €",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Chiffre d'affaires : %{y:,.0f} €<extra></extra>"
)

fig.update_layout(
    xaxis_title="Sexe",
    yaxis_title="Chiffre d'affaires (€)",
    yaxis_range=[0, ca_sexe.max() * 1.15]
)

fig.show()

### Interprétation

Le chiffre d'affaires généré par les clients masculins est légèrement supérieur à celui des clientes.

Les hommes réalisent un chiffre d'affaires d'environ **6,08 millions d'euros**, contre **5,95 millions d'euros** pour les femmes.

L'écart entre les deux sexes reste toutefois limité (environ **126 000 €**, soit près de **2 %** du chiffre d'affaires), ce qui montre que les contributions des clients masculins et féminins sont globalement équilibrées.

La librairie attire donc une clientèle relativement diversifiée, sans qu'un sexe domine fortement les revenus générés.

## 6.7 Chiffre d'affaires par tranche d'âge

Nous analysons le chiffre d'affaires généré selon les tranches d'âge des clients afin d'identifier les segments de clientèle les plus rentables.

In [ ]:
transactions_final["age"] = 2023 - transactions_final["birth"]

transactions_final[["birth","age"]].head()

In [ ]:
# Création des tranches d'âge

bins = [0,18,25,35,50,65,100]

labels = [
    "Moins de 18 ans",
    "18-25 ans",
    "26-35 ans",
    "36-50 ans",
    "51-65 ans",
    "Plus de 65 ans"
]

transactions_final["tranche_age"] = pd.cut(
    transactions_final["age"],
    bins=bins,
    labels=labels
)

transactions_final[["age","tranche_age"]].head()

In [ ]:
ca_age = (
    transactions_final
    .groupby("tranche_age", observed=False)["price"]
    .sum()
)

ca_age

In [ ]:
import plotly.express as px

fig = px.bar(
    x=ca_age.index.astype(str),
    y=ca_age.values,
    text=ca_age.values,
    labels={
        "x": "Tranche d'âge",
        "y": "Chiffre d'affaires (€)"
    },
    title="Chiffre d'affaires par tranche d'âge"
)

fig.update_traces(
    texttemplate="%{text:,.0f} €",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Chiffre d'affaires : %{y:,.0f} €<extra></extra>"
)

fig.update_layout(
    xaxis_title="Tranche d'âge",
    yaxis_title="Chiffre d'affaires (€)",
    yaxis_range=[0, ca_age.max() * 1.15]
)

fig.show()

### Interprétation

Le chiffre d'affaires varie selon les tranches d'âge des clients.

Les clients âgés de **36 à 50 ans** génèrent le chiffre d'affaires le plus élevé, avec près de **4,38 millions d'euros**, ce qui en fait la clientèle la plus rentable.

Les tranches **18 à 25 ans**, **26 à 35 ans** et **51 à 65 ans** contribuent également de manière importante, avec un chiffre d'affaires compris entre **2,0 et 2,3 millions d'euros**.

Les clients de **plus de 65 ans** génèrent un chiffre d'affaires plus faible, tandis qu'aucune vente n'a été enregistrée pour les clients de **moins de 18 ans**.

Ces résultats montrent que l'activité de la librairie repose principalement sur une clientèle adulte, en particulier les personnes âgées de **36 à 50 ans**.

## 6.8 — Nombre de ventes par sexe avec Plotly Express.

In [ ]:
ventes_sexe = (
    transactions_final
    .groupby("sex")
    .size()
)

ventes_sexe

In [ ]:
import plotly.express as px

# Renommer les modalités pour améliorer la lisibilité
labels_sexe = {
    "f": "Femmes",
    "m": "Hommes"
}

fig = px.bar(
    x=ventes_sexe.index.map(labels_sexe),
    y=ventes_sexe.values,
    text=ventes_sexe.values,
    labels={
        "x": "Sexe",
        "y": "Nombre de ventes"
    },
    title="Nombre de ventes par sexe"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Nombre de ventes : %{y:,.0f}<extra></extra>"
)

fig.update_layout(
    xaxis_title="Sexe",
    yaxis_title="Nombre de ventes",
    yaxis_range=[0, ventes_sexe.max() * 1.15]
)

fig.show()

### Interprétation

Le nombre de ventes est presque parfaitement équilibré entre les femmes et les hommes.

Les **hommes** représentent **344 841 ventes**, contre **342 693 ventes** pour les **femmes**. L'écart entre les deux groupes est donc très faible.

Cette répartition confirme l'observation réalisée sur le chiffre d'affaires par sexe : les hommes et les femmes contribuent de manière très similaire à l'activité commerciale de la librairie.

Le sexe des clients ne semble donc pas constituer, à lui seul, un facteur fortement différenciant du volume de ventes ou du chiffre d'affaires.

## 6.9 Nombre de ventes par tranche d'âge

Nous analysons le nombre de ventes réalisées par chaque tranche d'âge afin d'identifier les groupes de clients les plus actifs en termes de volume d'achats.

Cette analyse complète l'étude du chiffre d'affaires par tranche d'âge et permet de mieux comprendre la contribution de chaque segment de clientèle à l'activité commerciale de la librairie.

In [ ]:
# Calcul du nombre de ventes par tranche d'âge

ventes_age = (
    transactions_final
    .groupby("tranche_age", observed=False)
    .size()
)

ventes_age

In [ ]:
import plotly.express as px

fig = px.bar(
    x=ventes_age.index.astype(str),
    y=ventes_age.values,
    text=ventes_age.values,
    labels={
        "x": "Tranche d'âge",
        "y": "Nombre de ventes"
    },
    title="Nombre de ventes par tranche d'âge"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Nombre de ventes : %{y:,.0f}<extra></extra>"
)

fig.update_layout(
    xaxis_title="Tranche d'âge",
    yaxis_title="Nombre de ventes",
    yaxis_range=[0, ventes_age.max() * 1.15]
)

fig.show()

### Interprétation

La tranche d'âge des **36-50 ans** est de loin la plus active, avec **330 099 ventes**. Elle constitue donc le principal segment de clientèle en termes de volume d'achats.

Les **51-65 ans** arrivent en deuxième position avec **135 005 ventes**, suivis des **26-35 ans** avec **109 560 ventes**.

Les **plus de 65 ans** enregistrent **63 890 ventes**, tandis que les **18-25 ans** représentent **48 980 ventes**.

Aucune vente n'est enregistrée pour les **moins de 18 ans**, ce qui indique qu'aucun client de cette tranche d'âge n'est présent dans les données analysées.

Cette analyse confirme les résultats observés précédemment sur le chiffre d'affaires par tranche d'âge : les **36-50 ans constituent le cœur de la clientèle de la librairie**, puisqu'ils dominent à la fois en **nombre de ventes** et en **chiffre d'affaires**.

# 7. Analyse des relations entre les variables

Après avoir étudié l'activité commerciale et le profil des clients à l'aide d'analyses descriptives, nous cherchons maintenant à identifier les éventuelles relations entre les différentes variables.

Cette partie permettra notamment d'étudier :

- le lien entre le sexe des clients et les catégories de produits achetées ;
- le lien entre l'âge des clients et le montant de leurs achats ;
- la relation entre l'âge des clients et les catégories de produits achetées.

Des méthodes statistiques et des visualisations seront utilisées afin de mesurer et d'interpréter ces relations.

## 7.1 Relation entre le sexe et la catégorie de produits

Nous analysons la répartition des achats par catégorie selon le sexe.

**Important :** ce test (et tout le volet comportemental) se fait **hors clients BtoB**.  
Les 4 clients professionnels sont exclus ici ; le détail de leur identification est aussi présenté en section 9.


In [ ]:
# Exclusion des 4 clients BtoB avant le khi²
B2B = ['c_1609', 'c_4958', 'c_6714', 'c_3454']
transactions_stats = transactions_final[
    ~transactions_final['client_id'].isin(B2B)
].copy()

table_sexe_categorie = pd.crosstab(
    transactions_stats['sex'],
    transactions_stats['categ']
)
table_sexe_categorie


In [ ]:
# Préparation des données pour la visualisation

table_sexe_categorie_plot = table_sexe_categorie.copy()

table_sexe_categorie_plot.index = ["Femmes", "Hommes"]
table_sexe_categorie_plot.columns = [
    "Catégorie 0",
    "Catégorie 1",
    "Catégorie 2"
]

# Transformation du tableau pour Plotly
table_sexe_categorie_plot = (
    table_sexe_categorie_plot
    .reset_index()
    .rename(columns={"index": "Sexe"})
    .melt(
        id_vars="Sexe",
        var_name="Catégorie",
        value_name="Nombre de ventes"
    )
)

table_sexe_categorie_plot

In [ ]:
# Visualisation interactive du nombre de ventes
# par sexe et par catégorie

fig = px.bar(
    table_sexe_categorie_plot,
    x="Catégorie",
    y="Nombre de ventes",
    color="Sexe",
    barmode="group",
    text="Nombre de ventes",
    title="Nombre de ventes par catégorie et par sexe",
    labels={
        "Catégorie": "Catégorie de produits",
        "Nombre de ventes": "Nombre de ventes"
    }
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Catégorie de produits",
    yaxis_title="Nombre de ventes",
    legend_title="Sexe",
    height=550
)

fig.show()

### Interprétation

La répartition du nombre de ventes par catégorie est globalement similaire entre les femmes et les hommes.

- La **catégorie 0** concentre le plus grand nombre de ventes pour les deux sexes, avec **206 103 ventes chez les femmes** et **209 356 ventes chez les hommes**.
- La **catégorie 1** arrive en deuxième position, avec **119 307 ventes chez les femmes** contre **116 285 chez les hommes**.
- La **catégorie 2** représente le plus faible volume de ventes, avec **17 283 ventes chez les femmes** et **19 200 chez les hommes**.

À première vue, les comportements d'achat des femmes et des hommes semblent donc relativement proches en termes de répartition entre les catégories de produits. Toutefois, cette observation visuelle ne permet pas à elle seule de conclure à l'absence de relation entre le sexe et la catégorie de produits achetée.

Un **test du Chi² d'indépendance** sera donc réalisé afin de déterminer si la relation entre ces deux variables est statistiquement significative.

### 7.1.1 Test du Chi² d'indépendance

Le test du Chi² d'indépendance permet de déterminer s'il existe une association statistiquement significative entre deux variables qualitatives.

Dans notre cas :

- **H₀ (hypothèse nulle)** : le sexe et la catégorie de produits achetée sont indépendants.
- **H₁ (hypothèse alternative)** : il existe une relation entre le sexe et la catégorie de produits achetée.

Le seuil de significativité retenu est de **5 % (α = 0,05)**.

In [ ]:
# Test du Chi² d'indépendance entre le sexe et la catégorie

from scipy.stats import chi2_contingency

chi2, p_value, ddl, effectifs_theoriques = chi2_contingency(
    table_sexe_categorie
)

print(f"Statistique du Chi² : {chi2:.2f}")
print(f"P-value : {p_value:.6f}")
print(f"Degrés de liberté : {ddl}")

### Interprétation du test du Chi²

Le test du Chi² (hors BtoB) donne une statistique d'environ **22,7** et une **p-value < 0,05**.

Nous rejetons H₀ d'indépendance : il existe une association statistiquement significative entre le sexe et la catégorie.

Cependant, avec un très grand nombre de transactions, même un écart minime devient « significatif ». Il faut donc mesurer l'**intensité** du lien avec le **V de Cramér**.


In [ ]:
# Calcul du V de Cramér

import numpy as np

n = table_sexe_categorie.to_numpy().sum()
min_dimension = min(table_sexe_categorie.shape) - 1

v_cramer = np.sqrt(
    chi2 / (n * min_dimension)
)

print(f"V de Cramér : {v_cramer:.4f}")


### Interprétation du V de Cramér

Le **V de Cramér** mesure la force de l'association entre deux variables qualitatives.

Hors BtoB, on obtient **V ≈ 0,006** (quasi nul) :

- **0** : aucune association ;
- **0 à 0,10** : association très faible ;
- **0,10 à 0,30** : association faible ;
- **0,30 à 0,50** : association modérée ;
- **> 0,50** : association forte.

**Conclusion business :** le sexe n'a pratiquement **aucun pouvoir explicatif** sur la catégorie achetée (significatif ≠ important).


In [ ]:
print('Corrélation ligne à ligne non retenue — analyses au niveau client en section 10.')


# 8. Contrôle d'intégrité des données

Avant de réaliser les analyses statistiques, il est nécessaire de vérifier la cohérence entre les différents jeux de données.

Ces contrôles permettent notamment d'identifier :

- les produits du catalogue jamais vendus ;
- les produits présents dans les ventes mais absents du catalogue ;
- les clients inscrits n'ayant jamais réalisé d'achat ;
- les clients présents dans les ventes mais absents du fichier clients.

Cette étape garantit que les analyses reposent sur une base de données cohérente.

In [ ]:
# Produits du catalogue jamais vendus

produits_non_vendus = products[
    ~products["id_prod"].isin(transactions["id_prod"])
]

print("Nombre de produits jamais vendus :", len(produits_non_vendus))

produits_non_vendus.head()

In [ ]:
# Produits présents dans les ventes mais absents du catalogue

produits_absents_catalogue = transactions[
    ~transactions["id_prod"].isin(products["id_prod"])
]

print("Produits absents du catalogue :", len(produits_absents_catalogue))

produits_absents_catalogue.head()

In [ ]:
# Clients sans achat

clients_sans_achat = customers[
    ~customers["client_id"].isin(transactions["client_id"])
]

print("Nombre de clients sans achat :", len(clients_sans_achat))

clients_sans_achat.head()

In [ ]:
# Clients présents dans les ventes mais absents du fichier clients

clients_inconnus = transactions[
    ~transactions["client_id"].isin(customers["client_id"])
]

print("Nombre de clients inconnus :", len(clients_inconnus))

clients_inconnus.head()

### Interprétation

Les contrôles d'intégrité montrent que :

- 21 produits du catalogue n'ont jamais été vendus sur la période étudiée ;
- aucun produit présent dans les ventes n'est absent du catalogue ;
- 21 clients inscrits n'ont effectué aucun achat ;
- aucun client présent dans les ventes n'est absent du fichier clients.

Ces vérifications confirment que les données utilisées pour les analyses sont cohérentes et qu'aucune anomalie de référencement n'a été détectée dans les transactions.

# 9. Identification des clients atypiques (BtoB)

Avant les analyses comportementales (Julie), on identifie les clients atypiques.

En classant les clients par CA, **4 clients** se détachent radicalement (le 4ᵉ ≈ 114 k€, le 5ᵉ ≈ 5 k€). Ce ne sont pas des particuliers mais des **professionnels (BtoB)**.

**Décision méthodologique :**
- on les **conserve** pour les analyses de CA (Annabelle) ;
- on les **exclut** des analyses comportementales / corrélations (Julie).

> On n'utilise **pas** l'IQR ici : il exclurait ~250 bons clients BtoC. On retient uniquement la rupture nette des 4 plus gros.


In [ ]:
# CA par client, trié pour visualiser la rupture BtoB
ca_client = (
    transactions_final
    .groupby('client_id')['price']
    .agg(chiffre_affaires='sum', nombre_achats='count')
    .sort_values('chiffre_affaires', ascending=False)
    .reset_index()
)
display(ca_client.head(6))


In [ ]:
print("Nombre de clients :", len(ca_client))

In [ ]:
ca_client["chiffre_affaires"].describe()

## 9.1 Identification des 4 clients BtoB


In [ ]:
import plotly.express as px

fig = px.line(
    x=list(range(1, 31)),
    y=ca_client['chiffre_affaires'].head(30),
    markers=True,
    title="CA des 30 premiers clients : rupture nette après le 4ᵉ",
    labels={'x': 'Rang du client', 'y': 'CA (€)'}
)
fig.add_vline(x=4.5, line_dash='dash', line_color='crimson')
fig.show()


In [ ]:
# Les 4 clients BtoB (rupture nette dans le classement CA)
B2B = ['c_1609', 'c_4958', 'c_6714', 'c_3454']

ca_total = transactions_final['price'].sum()
ca_b2b = transactions_final.loc[transactions_final['client_id'].isin(B2B), 'price'].sum()

print('Clients BtoB :', B2B)
print(f'CA total     : {ca_total:,.0f} €')
print(f'CA BtoB      : {ca_b2b:,.0f} € ({ca_b2b/ca_total*100:.1f}% du CA)')


In [ ]:
clients_btob = ca_client[ca_client['client_id'].isin(B2B)]
print('Nombre de clients BtoB :', len(clients_btob))
display(clients_btob)


## 9.2 Création d'une base sans clients BtoB


In [ ]:
# Liste des 4 clients BtoB uniquement
liste_btob = B2B


In [ ]:
# Base utilisée pour les analyses statistiques (Julie)
transactions_stats = transactions_final[
    ~transactions_final['client_id'].isin(liste_btob)
].copy()

print('Transactions initiales :', len(transactions_final))
print('Transactions hors BtoB :', len(transactions_stats))
print('Clients BtoB restants  :', transactions_stats['client_id'].isin(B2B).sum(), '(doit valoir 0)')
transactions_stats.head()


In [ ]:
print('Transactions initiales :', len(transactions_final))
print('Transactions après exclusion des 4 BtoB :', len(transactions_stats))


In [ ]:
print('Clients retirés (BtoB) :', len(liste_btob))
print('Clients conservés :', transactions_stats['client_id'].nunique())


### Interprétation

En classant les clients par CA, **4 clients professionnels (BtoB)** se détachent clairement : `c_1609`, `c_4958`, `c_6714`, `c_3454`.

Ils représentent environ **7,4 % du CA** total alors qu'ils ne sont que 4 sur ~8 600 clients. Le 5ᵉ client est ~20× plus petit que le 4ᵉ : la rupture est nette.

On **ne retient pas l'IQR** (qui aurait exclu ~251 clients) : trop large, il supprimerait de nombreux bons clients particuliers.

Conformément à la méthodologie du projet :
- ces 4 clients restent dans les analyses de **chiffre d'affaires** ;
- ils sont **exclus** des analyses de **comportement client** (corrélations Julie), via `transactions_stats`.


## 10. Construction de la base d'analyse client

Les analyses statistiques seront réalisées au niveau du client et non plus au niveau de chaque ligne de transaction.

Pour chaque client, nous calculons :

- son chiffre d'affaires total ;
- son nombre de commandes ;
- son panier moyen ;
- son âge ;
- son sexe ;
- sa tranche d'âge.

Cette table servira de base à toutes les analyses comportementales.

In [ ]:
# Construction de la base d'analyse client

clients_stats = (
    transactions_stats
    .groupby("client_id", as_index=False)
    .agg(
        chiffre_affaires=("price", "sum"),
        nombre_commandes=("session_id", "nunique"),
        age=("age", "first"),
        sex=("sex", "first"),
        tranche_age=("tranche_age", "first")
    )
)

clients_stats.head()

In [ ]:
# Calcul du panier moyen par client

clients_stats["panier_moyen"] = (
    clients_stats["chiffre_affaires"]
    / clients_stats["nombre_commandes"]
)

clients_stats.head()

In [ ]:
print("Nombre de clients :", len(clients_stats))

clients_stats.describe()

### Interprétation

La base d'analyse comporte **8 596 clients**, après exclusion des **4 clients BtoB**.

Les statistiques descriptives montrent la distribution de l'âge, du chiffre d'affaires, de la fréquence d'achat et du panier moyen. Cette table client devient l'unité d'analyse pour toutes les corrélations suivantes.


## 10.1 Corrélation entre l'âge et le chiffre d'affaires par client

Cette analyse vise à déterminer si le chiffre d'affaires généré par un client est lié à son âge.

Les calculs sont réalisés sur la base des clients (et non plus des transactions), après exclusion des clients atypiques.

In [ ]:
from scipy.stats import pearsonr

coef_pearson, p_pearson = pearsonr(
    clients_stats["age"],
    clients_stats["chiffre_affaires"]
)

print(f"Coefficient de Pearson : {coef_pearson:.4f}")
print(f"P-value : {p_pearson:.6f}")

In [ ]:
from scipy.stats import spearmanr

coef_spearman, p_spearman = spearmanr(
    clients_stats["age"],
    clients_stats["chiffre_affaires"]
)

print(f"Coefficient de Spearman : {coef_spearman:.4f}")
print(f"P-value : {p_spearman:.6f}")

In [ ]:
import plotly.express as px

fig = px.scatter(
    clients_stats,
    x="age",
    y="chiffre_affaires",
    opacity=0.5,
    trendline="ols",
    title="Relation entre l'âge et le chiffre d'affaires par client",
    labels={
        "age": "Âge",
        "chiffre_affaires": "Chiffre d'affaires (€)"
    }
)

fig.show()

### Interprétation

Le test de Pearson met en évidence une corrélation négative faible entre l'âge du client et son chiffre d'affaires cumulé **(r = -0,178 ; p < 0,001)**.

Le test de Spearman aboutit à une conclusion similaire **(ρ = -0,174 ; p < 0,001 )**,confirmant qu'il existe une légère tendance selon laquelle les clients les plus jeunes génèrent un chiffre d'affaires plus élevé que les clients plus âgés.

Le nuage de points illustre cette tendance générale, bien que les observations restent fortement dispersées.

Ainsi, même si la relation est statistiquement significative grâce à la taille importante de l'échantillon, son intensité demeure faible. L'âge n'explique donc qu'une faible part des différences de chiffre d'affaires entre les clients.

## 10.2 Corrélation entre l'âge et la fréquence d'achat

In [ ]:
from scipy.stats import pearsonr

coef_pearson_freq, p_pearson_freq = pearsonr(
    clients_stats["age"],
    clients_stats["nombre_commandes"]
)

print(f"Coefficient de Pearson : {coef_pearson_freq:.4f}")
print(f"P-value : {p_pearson_freq:.6f}")

In [ ]:
from scipy.stats import spearmanr

coef_spearman_freq, p_spearman_freq = spearmanr(
    clients_stats["age"],
    clients_stats["nombre_commandes"]
)

print(f"Coefficient de Spearman : {coef_spearman_freq:.4f}")
print(f"P-value : {p_spearman_freq:.6f}")

In [ ]:
import plotly.express as px

fig = px.scatter(
    clients_stats,
    x="age",
    y="nombre_commandes",
    opacity=0.6,
    trendline="ols",
    title="Relation entre l'âge et la fréquence d'achat"
)

fig.update_layout(
    xaxis_title="Âge",
    yaxis_title="Nombre de commandes"
)

fig.show()

### Interprétation

Le test de Pearson met en évidence une corrélation positive faible entre l'âge du client et sa fréquence d'achat **(r = 0,206 ; p < 0,001)**.

Le test de Spearman confirme cette tendance **(ρ = 0,232 ; p < 0,001)**, indiquant qu'en moyenne les clients les plus âgés effectuent légèrement plus de commandes que les plus jeunes.

Le nuage de points illustre cette tendance générale, tout en montrant une forte dispersion des observations.

Ainsi, bien que la relation soit statistiquement significative, son intensité demeure faible. L'âge influence donc légèrement la fréquence d'achat, mais d'autres facteurs expliquent davantage les différences de comportement entre les clients.

## 10.3 Corrélation entre l'âge et le panier moyen

In [ ]:
from scipy.stats import pearsonr

coef_pearson_panier, p_pearson_panier = pearsonr(
    clients_stats["age"],
    clients_stats["panier_moyen"]
)

print(f"Coefficient de Pearson : {coef_pearson_panier:.4f}")
print(f"P-value : {p_pearson_panier:.6f}")

In [ ]:
from scipy.stats import spearmanr

coef_spearman_panier, p_spearman_panier = spearmanr(
    clients_stats["age"],
    clients_stats["panier_moyen"]
)

print(f"Coefficient de Spearman : {coef_spearman_panier:.4f}")
print(f"P-value : {p_spearman_panier:.6f}")

In [ ]:
import plotly.express as px

fig = px.scatter(
    clients_stats,
    x="age",
    y="panier_moyen",
    opacity=0.6,
    trendline="ols",
    title="Relation entre l'âge et le panier moyen par client"
)

fig.update_layout(
    xaxis_title="Âge",
    yaxis_title="Panier moyen (€)"
)

fig.show()

### Interprétation

Le test de Pearson évalue la relation linéaire entre l'âge du client et son panier moyen.

Le test de Spearman permet de confirmer cette relation en tenant compte d'une éventuelle relation monotone non linéaire.

Le nuage de points permet de visualiser la dispersion des observations et la tendance générale.

La conclusion sera formulée à partir des coefficients obtenus et de leur significativité statistique.

### Interprétation

Le test de Pearson met en évidence une corrélation négative modérée entre l'âge du client et son panier moyen **(r = -0,617 ; p < 0,001)**.

Le test de Spearman confirme cette relation avec une intensité encore plus marquée **(ρ = -0,700 ; p < 0,001)**, ce qui indique une forte relation monotone décroissante.

Le nuage de points montre que les clients les plus jeunes réalisent généralement des paniers moyens plus élevés, tandis que le montant moyen dépensé par commande diminue progressivement avec l'âge.

Cette différence entre les coefficients de Pearson et de Spearman suggère que la relation n'est pas parfaitement linéaire, mais suit une tendance monotone bien marquée.

Ce résultat constitue l'un des principaux enseignements de cette étude : les jeunes clients dépensent davantage à chaque commande, alors que les clients plus âgés effectuent légèrement plus de commandes, mais pour des montants unitaires plus faibles.

## 10.4 Influence de la catégorie de produits sur l'âge des clients

On compare l'âge entre catégories. L'ANOVA exige l'homogénéité des variances (test de **Levene**).  
Si Levene rejette H₀, on retient **Kruskal-Wallis**. L'intensité se mesure par **η²**.


In [ ]:
# Âge des clients selon la catégorie de produit

groupe0 = transactions_stats.loc[
    transactions_stats["categ"] == 0,
    "age"
]

groupe1 = transactions_stats.loc[
    transactions_stats["categ"] == 1,
    "age"
]

groupe2 = transactions_stats.loc[
    transactions_stats["categ"] == 2,
    "age"
]

In [ ]:
from scipy.stats import levene, f_oneway, kruskal
import numpy as np

# 1) Condition ANOVA : égalité des variances
W, p_levene = levene(groupe0, groupe1, groupe2)
print(f'Levene : W = {W:.1f} | p = {p_levene:.2e}')

# 2) ANOVA (référence) + Kruskal-Wallis (non paramétrique)
F, p_anova = f_oneway(groupe0, groupe1, groupe2)
H, p_kruskal = kruskal(groupe0, groupe1, groupe2)
print(f'ANOVA          : F = {F:.1f} | p = {p_anova:.2e}')
print(f'Kruskal-Wallis : H = {H:.1f} | p = {p_kruskal:.2e}')

if p_levene < 0.05:
    print("-> Variances inégales : on RETIENT Kruskal-Wallis (pas l'ANOVA).")

# 3) Mesure d'effet η²
ages = transactions_stats['age']
grand = ages.mean()
groupes = [groupe0, groupe1, groupe2]
ss_inter = sum(len(g) * (g.mean() - grand)**2 for g in groupes)
ss_total = ((ages - grand)**2).sum()
eta2 = ss_inter / ss_total
print(f'\nη² (rapport de corrélation) = {eta2:.3f}')


In [ ]:
age_categorie = (
    transactions_stats
    .groupby("categ")["age"]
    .agg(["count", "mean", "median", "std"])
)

age_categorie

In [ ]:
import plotly.express as px

fig = px.box(
    transactions_stats,
    x="categ",
    y="age",
    points=False,
    title="Distribution de l'âge selon la catégorie de produits"
)

fig.update_layout(
    xaxis_title="Catégorie",
    yaxis_title="Âge"
)

fig.show()

### Interprétation

Le test de **Levene** rejette l'égalité des variances → on retient **Kruskal-Wallis** (et non l'ANOVA).

Kruskal-Wallis est significatif (**p < 0,001**). L'effet est **modéré** (**η² ≈ 0,11**) :

- **catégorie 2** : clients jeunes (âge moyen ≈ 27 ans) ;
- **catégorie 0** : ≈ 45 ans ;
- **catégorie 1** : ≈ 49 ans.

La catégorie de produits est clairement liée à l'âge : levier marketing pertinent.


In [ ]:
import numpy as np

# Chiffre d'affaires des clients (hors BtoB)
ca = np.sort(clients_stats["chiffre_affaires"])

ca[:10]

In [ ]:
# Somme cumulée

ca_cumule = np.cumsum(ca)

# Normalisation

lorenz = np.insert(ca_cumule / ca_cumule[-1], 0, 0)

# Part de la population

x = np.linspace(0, 1, len(lorenz))

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=x,
        y=lorenz,
        mode="lines",
        name="Courbe de Lorenz"
    )
)

fig.add_trace(
    go.Scatter(
        x=[0,1],
        y=[0,1],
        mode="lines",
        name="Égalité parfaite",
        line=dict(dash="dash")
    )
)

fig.update_layout(
    title="Courbe de Lorenz du chiffre d'affaires par client",
    xaxis_title="Part cumulée des clients",
    yaxis_title="Part cumulée du chiffre d'affaires",
    template="plotly_white"
)

fig.show()



In [ ]:
aire = np.trapezoid(lorenz, x)

gini = 1 - 2 * aire

print(f"Indice de Gini : {gini:.4f}")

## Interprétation

**La courbe de Lorenz** montre que **le chiffre d'affaires n'est pas réparti de manière parfaitement uniforme entre les clients**.

**L'indice de Gini** est égal à **0,3795**, ce qui traduit une concentration modérée du chiffre d'affaires.

Autrement dit, une partie des clients génère une proportion plus importante des ventes, sans pour autant représenter une concentration excessive.

Cette analyse met en évidence l'existence de **clients plus fidèles ou plus dépensiers**, tout en montrant que l'activité commerciale repose sur une clientèle relativement diversifiée.

Ces résultats sont utiles pour identifier les segments de clients à forte valeur et adapter les actions de fidélisation.

## 11. Top / Flop des références

Un « top » se lit de deux façons : par **CA** ou par **quantité**. Les classements diffèrent.


In [ ]:
ref = (
    transactions_final
    .groupby('id_prod')
    .agg(nombre_ventes=('id_prod', 'count'), chiffre_affaires=('price', 'sum'))
    .join(products.set_index('id_prod')[['categ', 'price']])
)

top_ca  = ref.sort_values('chiffre_affaires', ascending=False).head(10)
top_qte = ref.sort_values('nombre_ventes', ascending=False).head(10)
flop_ca = ref.sort_values('chiffre_affaires').head(10)

print('--- TOP 10 par CA ---'); display(top_ca)
print('--- TOP 10 par QUANTITÉ ---'); display(top_qte)
print('--- FLOP 10 par CA ---'); display(flop_ca)

print('\nCatégories des tops CA :'); print(top_ca['categ'].value_counts().to_string())
print('Catégories des tops quantité :'); print(top_qte['categ'].value_counts().to_string())
print('Références jamais vendues :', len(set(products['id_prod']) - set(transactions_final['id_prod'])))


In [ ]:
# Flops déjà affichés ci-dessus (flop_ca)
flop_ca


In [ ]:
import plotly.express as px

fig = px.bar(
    top_ca.reset_index(),
    x='id_prod', y='chiffre_affaires',
    title='Top 10 références par chiffre_affaires',
    labels={'chiffre_affaires': 'chiffre_affaires(€)', 'id_prod': 'Référence'}
)
fig.show()


In [ ]:
fig = px.bar(
    top_qte.reset_index(),
    x='id_prod', y='nombre_ventes',
    title='Top 10 références par quantité',
    labels={'nombre_ventes': 'Quantité', 'id_prod': 'Référence'}
)
fig.show()


## Interprétation

- **Tops par CA** : surtout **catégorie 2** (produits chers, volumes modestes).
- **Tops par quantité** : surtout **catégories 0 et 1** (produits accessibles, forts volumes).
- **Flops** : références à très bas CA, souvent vendues 1–2 fois ; s'y ajoutent les **21 références jamais vendues** (candidates au déréférencement).
